# HAM10000 ResNet Fine-tuning
This notebook demonstrates how to fine-tune a ResNet model on the HAM10000 dataset using the `ResNetFineTuneTrainer` class.

In [4]:
import sys
from pathlib import Path

current_dir = Path.cwd()
proj_dir = current_dir.parent.parent.resolve() # get project root directory

print(f"Project directory set to: {proj_dir}")

# Add the project root to the system path to import local modules
sys.path.append(str(proj_dir / 'python'))

Project directory set to: C:\Users\z004ps0b\Documents\dermascan


In [5]:
from dataset_manager import DatasetManager
from pipeline.resnet_finetune_train import ResNetFineTuneTrainer

## Prepare data
Initialize > Load > Split in train, val, test

In [6]:
# Initialize the dataset manager
manager = DatasetManager(data_dir=str(proj_dir / "data"))

# Load metadata (filtering for Melanoma vs Nevi as per trainer restrictions)
manager.load_cleaned_metadata(filter_melanoma_only=True)

# Split the data into train (70%), validation (15%), and test (15%) sets
train_df, val_df, test_df = manager.train_test_val_split(val_size=0.15, test_size=0.15)

print(f"Training samples: {len(train_df)}")
print(f"Validation samples: {len(val_df)}")
print(f"Test samples: {len(test_df)}")

Training samples: 5470
Validation samples: 1173
Test samples: 1173


## Configure ResNetFineTuneTrainer

In [7]:
trainer = ResNetFineTuneTrainer()

# Set up the model (ResNet50, 2 classes: nv vs mel)
trainer.set_model(model_name='resnet50', pretrained=True) # , input_channels=6

# Set data loaders
trainer.set_data_loaders(train_df, val_df, test_df, filter_name='lbp_filter', data_path=str(proj_dir / "data"))

# Configure optimizer and scheduler
trainer.configure_optimizer(optimizer_type='adam', lr=1e-4, weight_decay=1e-4)
trainer.configure_scheduler(scheduler_type='reduce_on_plateau', patience=3)

# Set output paths for artifacts
trainer.set_output_paths(
    model_save_dir=str(proj_dir / "python" / "results" / "models"),
    results_csv_path=str(proj_dir / "python" / "results" / "training_results.csv"),
    history_plot_dir=str(proj_dir / "python" / "results" / "plots")
)

## Train model

In [8]:
# Train for a specified number of epochs
trainer.train(epochs=1)

Epoch 1/1
----------


FileNotFoundError: Caught FileNotFoundError in DataLoader worker process 0.
Original Traceback (most recent call last):
  File "c:\Users\z004ps0b\Documents\dermascan\.venv\Lib\site-packages\torch\utils\data\_utils\worker.py", line 358, in _worker_loop
    data = fetcher.fetch(index)  # type: ignore[possibly-undefined]
           ^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\z004ps0b\Documents\dermascan\.venv\Lib\site-packages\torch\utils\data\_utils\fetch.py", line 54, in fetch
    data = [self.dataset[idx] for idx in possibly_batched_index]
            ~~~~~~~~~~~~^^^^^
  File "C:\Users\z004ps0b\Documents\dermascan\python\pipeline\dataset_loader.py", line 48, in __getitem__
    filtered_features = np.load(filtered_path / self.filter_name / (self.dataframe.iloc[idx]['image_id'] + ".npy"))
                        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\z004ps0b\Documents\dermascan\.venv\Lib\site-packages\numpy\lib\_npyio_impl.

## Evaluate and Save Results

In [ ]:
trainer.evaluate()
trainer.save_results_to_csv()

Evaluation results - F1 (macro): 0.4617, F1 (weighted): 0.7919, AUC: 0.3859, Accuracy: 0.8576, Sensitivity: 0.0000, Specificity: 1.0000
